In [1]:
import healpy as hp
from astropy.io import fits
from astropy import conf
from astropy.table import Table, vstack, hstack, join
from astropy.coordinates import Distance
from astropy.cosmology import Planck18, LambdaCDM
import numpy as np
import fitsio
import numpy.lib.recfunctions as rfn
import os
import pickle
import vast.catalog.void_catalog as void_catalog

## Load Galaxies

In [8]:
gal_dir = '/global/cfs/cdirs/desi/users/hrincon/DESIVAST/galaxy_catalog/'

# Angular mask of 100% Survey Completeness
#mask_file = f"{gal_dir}/mask/loa_mask.fits"
smoothed_mask_file = f"{gal_dir}/mask/loa_mask_smoothed.fits"

#smoothed_mask_file = '/pscratch/sd/h/hrincon/loa_mask_smoothed.fits'


# Redshift limits
zmin = 0.
zmax = 0.231

# Read in mask
# There are two mask versions. The first ("mask") covers all regions of BGS that have been completed (for up to 4 passes).
# The second ("smoothed_mask") removes small non-contiguous regions from the mask for the purpose of void-finding on a contigious mask
# From now on, we will use "mask" to refer to "smoothed mask"
mask=fitsio.read(smoothed_mask_file)
#smoothed_mask=fitsio.read(smoothed_mask_file)
nside=hp.npix2nside(len(mask)) #healpix nside parameter

def get_spec_file_name (healpix_index):
    # Fastspec fit files are split up into different healpix groups that cover differnt regions of the sky
    # There are 12 files total
    formated_index = str(healpix_index).zfill(2)
    
    file_name = f'/global/cfs/cdirs/desi/vac/dr2/fastspecfit/loa/v1.0/catalogs/fastspec-loa-main-bright-nside1-hp{formated_index}.fits'

    return file_name

# Save information about the mask and masked galaxies


ngc_catalog = Table(names = ['TARGETID','RA','DEC','Z', 'BGS_TARGET',
                          'ABSMAG01_SDSS_U','ABSMAG01_SDSS_G', 'ABSMAG01_SDSS_R', 'ABSMAG01_SDSS_Z',
                          'LOGMSTAR', 'SFR', 'HALPHA_EW'],
                    dtype =['>i8', '>f8', '>f8', '>f8', '>i8', '>f4', '>f4', '>f4', '>f4', '>f4', '>f4', '>f4']
)

sgc_catalog = Table(names = ['TARGETID','RA','DEC','Z', 'BGS_TARGET',
                          'ABSMAG01_SDSS_U','ABSMAG01_SDSS_G', 'ABSMAG01_SDSS_R', 'ABSMAG01_SDSS_Z',
                          'LOGMSTAR', 'SFR', 'HALPHA_EW'],
                   dtype =['>i8', '>f8', '>f8', '>f8', '>i8', '>f4', '>f4', '>f4', '>f4', '>f4', '>f4', '>f4']
)

bright_count = 0
redshift_count = 0
mask_count = 0
quality_count = 0

#Open the FastSpecFit VAC
for idx in range (12): #12 bgs files total
    
    with fitsio.FITS(get_spec_file_name(idx)) as full_catalog:

        
        print("reading data", idx)
        metadata = full_catalog[1]['TARGETID','Z','ZWARN','DELTACHI2','SPECTYPE','RA','DEC','BGS_TARGET', 'SURVEY', 'PROGRAM'
                           ][:]
        specphot = full_catalog[2][
                           'ABSMAG01_SDSS_U', 'ABSMAG01_SDSS_G',
                           'ABSMAG01_SDSS_R', 'ABSMAG01_SDSS_Z',
                           'LOGMSTAR', 'SFR', #'HALPHA_FLUX', 'HBETA_FLUX',
                          ][:]

        fastspec = full_catalog[3]['HALPHA_EW',][:]


        catalog = rfn.merge_arrays([metadata, specphot, fastspec], flatten=True, usemask=False)

        del metadata, specphot, fastspec

        print(len(catalog), "new target observations read in")
        
        # Select BGS Bright galaxies
        select = np.where(
                   (catalog['SPECTYPE']=='GALAXY') 
                   & (catalog['SURVEY']=='main')
                   & (catalog['PROGRAM']=='bright')
                 )
        
        catalog=catalog[select]
            
        #check for duplicate targets
        _, select = np.unique(catalog['TARGETID'], return_index=True)
        if len(catalog) != len(select):
            raise ValueError(f'Duplicate galaxies detected. {len(select)} out of {len(catalog)} are unique')
                
        bright_count += len(catalog[(catalog['BGS_TARGET'] & 2**1 != 0) + (catalog['BGS_TARGET'] & 2**0 != 0)])
        print(bright_count, "bright time galaxies")
    
        # Impose redshift limits
        print("Imposing redshift limits")
        select = np.where((catalog['Z']>zmin)  # > zmin and not >=zmin to avoid galaxies at origin
                        & (catalog['Z']<=zmax) 
                     )

        catalog=catalog[select]
        
        redshift_count += len(catalog[(catalog['BGS_TARGET'] & 2**1 != 0) + (catalog['BGS_TARGET'] & 2**0 != 0)])
        print(redshift_count, "galaxies in redshift limits")
        
        # Cut on mask
        #The pixel IDs for every (ra, dec) position
        pxid=hp.ang2pix(nside, catalog['RA'], catalog['DEC'], nest=True,lonlat=True)
        #Select the galaxies that fall within the mask
        select = np.isin(pxid, mask[mask["DONE"]==1]["HPXPIXEL"])
        catalog=catalog[select]

        mask_count += len(catalog[(catalog['BGS_TARGET'] & 2**1 != 0) + (catalog['BGS_TARGET'] & 2**0 != 0)])
        print(mask_count, "galaxies within smoothed angular mask")
        
        #Quality cuts 
        #made to match Ross 2024, The Dark Energy Spectroscopic Instrument: Construction of Large-scale Structure Catalogs
        select = np.where(
                   (catalog['ZWARN']==0) 
                   & (catalog['DELTACHI2']>40) 
        )   
        
        catalog=catalog[select]

        quality_count += len(catalog[(catalog['BGS_TARGET'] & 2**1 != 0) + (catalog['BGS_TARGET'] & 2**0 != 0)])
        print(quality_count, "galaxies in final catalog")
        
        
        #split into NGC and SGC
        select = (catalog['RA'] < 304) * (catalog['RA'] > 83)

        print(len(catalog),'galaxies in NGC')
        print(len(catalog),'galaxies in SGC')
        
        #save catalog
        out=Table([catalog['TARGETID'],
                   catalog['RA'],
                   catalog['DEC'],
                   catalog['Z'],
                   catalog['BGS_TARGET'],
                   catalog['ABSMAG01_SDSS_U'],
                   catalog['ABSMAG01_SDSS_G'],
                   catalog['ABSMAG01_SDSS_R'],
                   catalog['ABSMAG01_SDSS_Z'],
                   catalog['LOGMSTAR'], 
                   catalog['SFR'], 
                   catalog["HALPHA_EW"]
                  ],
                   
                   names=['TARGETID','RA','DEC','Z', 'BGS_TARGET',
                          'ABSMAG01_SDSS_U','ABSMAG01_SDSS_G', 'ABSMAG01_SDSS_R', 'ABSMAG01_SDSS_Z',
                          'LOGMSTAR', 'SFR', 'HALPHA_EW']
                 )

        ngc_catalog = vstack([ngc_catalog, out[select]])
        sgc_catalog = vstack([sgc_catalog, out[~select]])

reading data 0
325494 new target observations read in
319762 bright time galaxies
Imposing redshift limits
160404 galaxies in redshift limits
84669 galaxies within smoothed angular mask
83146 galaxies in final catalog
83436 galaxies in NGC
83436 galaxies in SGC
reading data 1
2693426 new target observations read in
2951451 bright time galaxies
Imposing redshift limits
1466279 galaxies in redshift limits
1131061 galaxies within smoothed angular mask
1112974 galaxies in final catalog
1035585 galaxies in NGC
1035585 galaxies in SGC
reading data 2
3141532 new target observations read in
6024915 bright time galaxies
Imposing redshift limits
2990850 galaxies in redshift limits
2505448 galaxies within smoothed angular mask
2462484 galaxies in final catalog
1357680 galaxies in NGC
1357680 galaxies in SGC
reading data 3
641198 new target observations read in
6656086 bright time galaxies
Imposing redshift limits
3294120 galaxies in redshift limits
2803290 galaxies within smoothed angular mask
27

In [9]:
bright_count,redshift_count,mask_count,quality_count

(12893539, 6390196, 5460691, 5365398)

In [10]:
# BGS Bright + Faint + other bright time surveys
len(ngc_catalog) + len(sgc_catalog)

5394272

In [11]:
# BGS Bright + Faint 
len(ngc_catalog[(ngc_catalog['BGS_TARGET'] & 2**1 != 0) + (ngc_catalog['BGS_TARGET'] & 2**0 != 0)]) + len(sgc_catalog[(sgc_catalog['BGS_TARGET'] & 2**1 != 0) + (sgc_catalog['BGS_TARGET'] & 2**0 != 0)])


5365398

In [12]:
# BGS Bright + Faint (NGC)
len(ngc_catalog[(ngc_catalog['BGS_TARGET'] & 2**1 != 0) + (ngc_catalog['BGS_TARGET'] & 2**0 != 0)])

4004867

In [13]:
# BGS Bright + Faint (SGC)
len(sgc_catalog[(sgc_catalog['BGS_TARGET'] & 2**1 != 0) + (sgc_catalog['BGS_TARGET'] & 2**0 != 0)])

1360531

## Enter Galaxies into Void Class

In [12]:
#import importlib
#importlib.reload(void_catalog)

<module 'vast.catalog.void_catalog' from '/global/u2/h/hrincon/VAST/python/vast/catalog/void_catalog.py'>

In [14]:
# in case anything goes wrong
ngc_backup = Table(ngc_catalog)
sgc_backup = Table(sgc_catalog)

In [15]:
ngc_catalog = Table(ngc_backup)
sgc_catalog = Table(sgc_backup)

In [16]:
vflags_exist = True
edge_buffer = 10 # Mpc/h

voidfinder_voids_path_ngc = '/global/cfs/cdirs/desi/users/hrincon/DESIVAST/voids/loa/minus20.0/DESIVAST_RedBlue_NGC_VoidFinder_Output.fits'
voidfinder_voids_path_sgc = '/global/cfs/cdirs/desi/users/hrincon/DESIVAST/voids/loa/minus20.0/DESIVAST_RedBlue_SGC_VoidFinder_Output.fits'
voidfinder_catalog = void_catalog.VoidFinderCatalogStacked(['N','S'], [voidfinder_voids_path_ngc, voidfinder_voids_path_sgc], edge_buffer=edge_buffer)

if vflags_exist:
    voidfinder_catalog.add_galaxies(['tmp','tmp'], [ngc_catalog, sgc_catalog], 
                                    [f'{gal_dir}/DR2_RB_N_VF_vflag.fits', f'{gal_dir}/DR2_RB_S_VF_vflag.fits'], 
                                    redshift_name='Z', ra_name = 'RA', dec_name='DEC'
                                   )
    ngc_catalog['VF_FLAG'] = voidfinder_catalog['N'].galaxies['vflag']
    sgc_catalog['VF_FLAG'] = voidfinder_catalog['S'].galaxies['vflag']
else:
    voidfinder_catalog.add_galaxies(['tmp','tmp'], [ngc_catalog, sgc_catalog], 
                                    redshift_name='Z', ra_name = 'RA', dec_name='DEC'
                                   )
    voidfinder_catalog.calculate_vflag([f'{gal_dir}/DR2_RB_N_VF_vflag.fits', f'{gal_dir}/DR2_RB_S_VF_vflag.fits'], num_cpus=35)
    ngc_catalog['VF_FLAG'] = voidfinder_catalog['N'].galaxies['vflag']
    sgc_catalog['VF_FLAG'] = voidfinder_catalog['S'].galaxies['vflag']

## Cut out Survey Edges

In [17]:
voidfinder_catalog['N'].galaxies['ABSMAG01_SDSS_R'].name='rabsmag'
voidfinder_catalog['S'].galaxies['ABSMAG01_SDSS_R'].name='rabsmag'

In [18]:
# SGC edge cut
sgc_edge_cut = voidfinder_catalog['S'].galaxy_membership(return_selector=True, mag_lim = np.inf)

In [19]:
# NGC edge cut
ngc_edge_cut = voidfinder_catalog['N'].galaxy_membership(return_selector=True, mag_lim = np.inf)

In [20]:
voidfinder_catalog['N'].galaxies['rabsmag'].name='ABSMAG01_SDSS_R'
voidfinder_catalog['S'].galaxies['rabsmag'].name='ABSMAG01_SDSS_R'

voidfinder_catalog['N'].galaxies['vflag'].name='VF_FLAG'
voidfinder_catalog['S'].galaxies['vflag'].name='VF_FLAG'

In [21]:
for name in ngc_catalog.colnames:
    ngc_catalog[name].name = str.upper(name)
for name in sgc_catalog.colnames:
    sgc_catalog[name].name = str.upper(name)

## Create Final Table

In [22]:
# in case anything goes wrong
ngc_backup = Table(ngc_catalog)
sgc_backup = Table(sgc_catalog)

In [31]:
#ngc_catalog=Table(ngc_backup)
#sgc_catalog=Table(sgc_backup)

In [34]:
ngc_catalog = ngc_catalog[ngc_edge_cut[0] + ngc_edge_cut[1]]
sgc_catalog = sgc_catalog[sgc_edge_cut[0] + sgc_edge_cut[1]]

In [35]:
# BGS Bright + Faint 
final_count = len(ngc_catalog[(ngc_catalog['BGS_TARGET'] & 2**1 != 0) + (ngc_catalog['BGS_TARGET'] & 2**0 != 0)]) + len(sgc_catalog[(sgc_catalog['BGS_TARGET'] & 2**1 != 0) + (sgc_catalog['BGS_TARGET'] & 2**0 != 0)])
# BGS Bright + Faint (NGC)
final_count_ngc = len(ngc_catalog[(ngc_catalog['BGS_TARGET'] & 2**1 != 0) + (ngc_catalog['BGS_TARGET'] & 2**0 != 0)])
# BGS Bright + Faint (SGC)
final_count_sgc = len(sgc_catalog[(sgc_catalog['BGS_TARGET'] & 2**1 != 0) + (sgc_catalog['BGS_TARGET'] & 2**0 != 0)])

In [36]:
# BGS Bright + Faint
final_count, final_count_ngc, final_count_sgc

(4612469, 3520845, 1091624)

In [37]:
# construct final galaxy table for analysis, using the edge cuts to remove edge galaxies
# BGS Bright + Faint + Other Bright Time Surveys
galaxy_table = vstack([ngc_catalog, sgc_catalog])
print(len(galaxy_table), len(ngc_catalog), len(sgc_catalog))
del ngc_catalog, sgc_catalog

4632835 3537870 1094965


In [38]:
#Save for later
outfile = '/global/cfs/cdirs/desi/users/hrincon/galaxy_environment/galaxy_table_DR2_voids.fits'
galaxy_table.write(outfile, format='fits', overwrite=True)